<a href="https://colab.research.google.com/github/JoseAroudo/Compiladores/blob/main/MiniTcl_expr_con_SLY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MiniTcl: implementación del comando `expr` con SLY

Este cuaderno transforma la idea del archivo `topdown.ipynb` en un componente utilizable por MiniTcl. En vez de escribir el parser manualmente, usamos **SLY** (`Lexer` + `Parser`), conservamos un **AST**, aplicamos el patrón **Visitor** y añadimos evaluación.

El subconjunto implementado acepta:

- números enteros y reales;
- variables MiniTcl escritas como `$nombre`;
- paréntesis y operadores unarios `+`, `-`, `!`;
- `*`, `/`, `%`, `+`, `-`;
- comparaciones `<`, `<=`, `>`, `>=`, `==`, `!=`;
- operadores lógicos `&&` y `||`;
- las formas `expr 1 + 2` y `expr {1 + 2}`.

> Alcance: es un `expr` educativo compatible con MiniTcl, no una reproducción total del `expr` de Tcl/Tk.

In [ ]:
# Ejecute esta celda una sola vez si SLY no está instalado.
!pip install -q sly

## 1. Arquitectura

El intérprete principal de MiniTcl reconoce el comando y entrega el texto que sigue a `expr`. A partir de allí este componente realiza cuatro pasos:

1. `ExprLexer` convierte caracteres en tokens.
2. `ExprParser` aplica la gramática y construye el AST.
3. `Evaluator` recorre el AST y obtiene un valor.
4. `MiniTclExpr` expone una interfaz simple para el intérprete.

La separación es importante: el lexer principal de MiniTcl **no debe intentar analizar internamente** toda la expresión. Solo debe preservar su texto y delegarlo a este módulo.

## 2. Nodos del AST

Una expresión como `1 + 2 * 3` no se evalúa durante el análisis sintáctico. Primero se representa como un árbol: `+(1, *(2, 3))`.

In [ ]:
from dataclasses import dataclass
from typing import Any, Mapping
from sly import Lexer, Parser


@dataclass(frozen=True)
class Node:
    pass


@dataclass(frozen=True)
class Number(Node):
    value: int | float


@dataclass(frozen=True)
class Variable(Node):
    name: str


@dataclass(frozen=True)
class Unary(Node):
    operator: str
    operand: Node


@dataclass(frozen=True)
class Binary(Node):
    operator: str
    left: Node
    right: Node

## 3. Lexer de expresiones

El token `VARIABLE` incluye el signo `$`. Esto reproduce la sustitución de variables que MiniTcl necesita dentro de `expr`. Los operadores de dos caracteres se declaran antes que los de un carácter para evitar reconocer `<=` como `<` seguido de `=`.

In [ ]:
class ExprLexer(Lexer):
    tokens = {
        NUMBER, VARIABLE,
        PLUS, MINUS, TIMES, DIVIDE, MOD,
        LT, LE, GT, GE, EQ, NE,
        AND, OR, NOT,
    }

    literals = {'(', ')'}
    ignore = ' \t'

    LE = r'<='
    GE = r'>='
    EQ = r'=='
    NE = r'!='
    AND = r'&&'
    OR = r'\|\|'
    PLUS = r'\+'
    MINUS = r'-'
    TIMES = r'\*'
    DIVIDE = r'/'
    MOD = r'%'
    LT = r'<'
    GT = r'>'
    NOT = r'!'
    VARIABLE = r'\$[A-Za-z_][A-Za-z0-9_]*'

    @_(r'(?:\d+\.\d*|\.\d+)(?:[eE][+-]?\d+)?|\d+[eE][+-]?\d+|\d+')
    def NUMBER(self, token):
        text = token.value
        token.value = float(text) if any(c in text for c in '.eE') else int(text)
        return token

    @_(r'\n+')
    def newline(self, token):
        self.lineno += token.value.count('\n')

    def error(self, token):
        raise SyntaxError(
            f"Carácter ilegal {token.value[0]!r} en la posición {self.index}"
        )

In [ ]:
lexer = ExprLexer()
[(tok.type, tok.value) for tok in lexer.tokenize('$x + 2 * (3 <= 4)')]

[('VARIABLE', '$x'),
 ('PLUS', '+'),
 ('NUMBER', 2),
 ('TIMES', '*'),
 ('(', '('),
 ('NUMBER', 3),
 ('LE', '<='),
 ('NUMBER', 4),
 (')', ')')]

## 4. Gramática de `expr`

Una forma compacta de la gramática es:

```ebnf
expression  ::= expression "||" expression
              | expression "&&" expression
              | expression ("==" | "!=") expression
              | expression ("<" | "<=" | ">" | ">=") expression
              | expression ("+" | "-") expression
              | expression ("*" | "/" | "%") expression
              | ("+" | "-" | "!") expression
              | "(" expression ")"
              | NUMBER
              | VARIABLE
```

Esta gramática es ambigua si se mira sola. SLY la desambigua mediante `precedence`: `||` tiene la menor prioridad; los operadores unarios, la mayor. Todos los binarios son asociativos por la izquierda.

In [ ]:
class ExprParser(Parser):
    tokens = ExprLexer.tokens

    precedence = (
        ('left', OR),
        ('left', AND),
        ('left', EQ, NE),
        ('left', LT, LE, GT, GE),
        ('left', PLUS, MINUS),
        ('left', TIMES, DIVIDE, MOD),
        ('right', UPLUS, UMINUS, NOT),
    )

    @_('expression OR expression',
       'expression AND expression',
       'expression EQ expression',
       'expression NE expression',
       'expression LT expression',
       'expression LE expression',
       'expression GT expression',
       'expression GE expression',
       'expression PLUS expression',
       'expression MINUS expression',
       'expression TIMES expression',
       'expression DIVIDE expression',
       'expression MOD expression')
    def expression(self, p):
        return Binary(p[1], p.expression0, p.expression1)

    @_('PLUS expression %prec UPLUS',
       'MINUS expression %prec UMINUS',
       'NOT expression')
    def expression(self, p):
        return Unary(p[0], p.expression)

    @_('"(" expression ")"')
    def expression(self, p):
        return p.expression

    @_('NUMBER')
    def expression(self, p):
        return Number(p.NUMBER)

    @_('VARIABLE')
    def expression(self, p):
        return Variable(p.VARIABLE[1:])

    def error(self, token):
        if token is None:
            raise SyntaxError('Expresión incompleta')
        raise SyntaxError(
            f"Token inesperado {token.value!r} en la línea {token.lineno}"
        )

## 5. Evaluador (Visitor)

En MiniTcl, los resultados lógicos se representan como `1` y `0`. `&&` y `||` usan evaluación de cortocircuito: el operando derecho solo se evalúa cuando hace falta.

In [ ]:
class ExprError(Exception):
    'Error controlado producido por expr.'


class Evaluator:
    def __init__(self, variables: Mapping[str, Any] | None = None):
        self.variables = dict(variables or {})

    def visit(self, node: Node):
        method = getattr(self, f'visit_{type(node).__name__}', None)
        if method is None:
            raise ExprError(f'No se puede evaluar {type(node).__name__}')
        return method(node)

    def visit_Number(self, node):
        return node.value

    def visit_Variable(self, node):
        if node.name not in self.variables:
            raise ExprError(f'Variable no definida: ${node.name}')
        value = self.variables[node.name]
        try:
            text = str(value)
            return float(text) if any(c in text for c in '.eE') else int(text)
        except (TypeError, ValueError):
            raise ExprError(f'El valor de ${node.name} no es numérico: {value!r}')

    def visit_Unary(self, node):
        value = self.visit(node.operand)
        operations = {
            '+': lambda x: +x,
            '-': lambda x: -x,
            '!': lambda x: int(not bool(x)),
        }
        return operations[node.operator](value)

    def visit_Binary(self, node):
        left = self.visit(node.left)

        # Cortocircuito: evita evaluar innecesariamente el lado derecho.
        if node.operator == '&&':
            return int(bool(left) and bool(self.visit(node.right)))
        if node.operator == '||':
            return int(bool(left) or bool(self.visit(node.right)))

        right = self.visit(node.right)
        operations = {
            '+': lambda a, b: a + b,
            '-': lambda a, b: a - b,
            '*': lambda a, b: a * b,
            '/': lambda a, b: a / b,
            '%': lambda a, b: a % b,
            '<': lambda a, b: int(a < b),
            '<=': lambda a, b: int(a <= b),
            '>': lambda a, b: int(a > b),
            '>=': lambda a, b: int(a >= b),
            '==': lambda a, b: int(a == b),
            '!=': lambda a, b: int(a != b),
        }
        try:
            return operations[node.operator](left, right)
        except ZeroDivisionError as exc:
            raise ExprError('División por cero') from exc

## 6. Fachada: el comando `expr`

La clase siguiente une lexer, parser y evaluador. También retira, de forma controlada, las llaves externas que MiniTcl usa para agrupar la expresión.

Ejemplos válidos:

```tcl
expr 1 + 2 * 3
expr {1 + 2 * 3}
set x 10
expr {$x >= 10 && $x < 20}
```

In [ ]:
class MiniTclExpr:
    def __init__(self):
        self.lexer = ExprLexer()

    @staticmethod
    def _unwrap(text: str) -> str:
        text = text.strip()
        if text.startswith('{') or text.endswith('}'):
            if not (text.startswith('{') and text.endswith('}')):
                raise ExprError('Llaves exteriores no balanceadas')
            text = text[1:-1].strip()
        if not text:
            raise ExprError('expr requiere una expresión')
        return text

    def ast(self, text: str) -> Node:
        source = self._unwrap(text)
        # Un parser nuevo evita conservar estado después de un error.
        return ExprParser().parse(self.lexer.tokenize(source))

    def evaluate(self, text: str, variables=None):
        tree = self.ast(text)
        return Evaluator(variables).visit(tree)


expr = MiniTclExpr()
expr.evaluate('{1 + 2 * 3 / 4 - 5}')

-2.5

In [ ]:
variables = {'x': 10, 'y': 3.5, 'cero': 0}

examples = [
    '1 + 2 * 3',
    '(1 + 2) * 3',
    '-$x + 4',
    '$x >= 10 && $y < 4',
    '$cero && (1 / 0)',       # demuestra el cortocircuito
    '{2 + 3 * 4 == 14}',
]

for source in examples:
    print(f'{source:25} -> {expr.evaluate(source, variables)}')

1 + 2 * 3                 -> 7
(1 + 2) * 3               -> 9
-$x + 4                   -> -6
$x >= 10 && $y < 4        -> 1
$cero && (1 / 0)          -> 0
{2 + 3 * 4 == 14}         -> 1


## 7. Visualización textual del AST

Esta versión no depende de Graphviz, de modo que funciona tanto en Jupyter Notebook como en Google Colab. La sangría muestra la estructura y permite comprobar la precedencia.

In [ ]:
class TreePrinter:
    def render(self, node: Node, prefix='', last=True):
        connector = '└── ' if last else '├── '
        label = self._label(node)
        lines = [prefix + connector + label]
        children = self._children(node)
        child_prefix = prefix + ('    ' if last else '│   ')
        for index, child in enumerate(children):
            lines.extend(self.render(child, child_prefix, index == len(children) - 1))
        return lines

    @staticmethod
    def _label(node):
        if isinstance(node, Number):
            return f'NUMBER {node.value}'
        if isinstance(node, Variable):
            return f'VARIABLE ${node.name}'
        if isinstance(node, Unary):
            return f'UNARY {node.operator}'
        return f'BINARY {node.operator}'

    @staticmethod
    def _children(node):
        if isinstance(node, Unary):
            return [node.operand]
        if isinstance(node, Binary):
            return [node.left, node.right]
        return []

    def __call__(self, node):
        return '\n'.join(self.render(node))


tree = expr.ast('$x + 2 * (3 - 1)')
print(TreePrinter()(tree))

└── BINARY +
    ├── VARIABLE $x
    └── BINARY *
        ├── NUMBER 2
        └── BINARY -
            ├── NUMBER 3
            └── NUMBER 1


## 8. Integración con el intérprete MiniTcl

El despachador de comandos puede conservar una tabla de variables y delegar en `MiniTclExpr`:

In [ ]:
class MiniTclDemo:
    'Integración mínima; no reemplaza el intérprete completo de MiniTcl.'

    def __init__(self):
        self.variables = {}
        self.expr_engine = MiniTclExpr()

    def execute(self, command: str):
        command = command.strip()
        if command.startswith('set '):
            _, name, value = command.split(maxsplit=2)
            self.variables[name] = value
            return value
        if command.startswith('expr '):
            expression_text = command[5:]
            return self.expr_engine.evaluate(expression_text, self.variables)
        raise ExprError('Comando desconocido')


mini = MiniTclDemo()
print(mini.execute('set x 8'))
print(mini.execute('expr {$x * 2 + 1}'))

8
17


## 9. Pruebas automáticas

Estas pruebas verifican precedencia, asociatividad, paréntesis, variables, comparaciones, lógica y errores.

In [ ]:
tests = [
    ('1 + 2 * 3', {}, 7),
    ('(1 + 2) * 3', {}, 9),
    ('10 - 3 - 2', {}, 5),
    ('-2 * 3', {}, -6),
    ('10 % 4', {}, 2),
    ('$a + 1', {'a': 9}, 10),
    ('3 < 4', {}, 1),
    ('3 == 4', {}, 0),
    ('1 || 0 && 0', {}, 1),
    ('!0', {}, 1),
]

for source, variables, expected in tests:
    actual = expr.evaluate(source, variables)
    assert actual == expected, (source, expected, actual)

error_tests = [
    ('1 / 0', {}, 'División por cero'),
    ('$ausente + 1', {}, 'Variable no definida'),
    ('1 +', {}, 'Expresión incompleta'),
]

for source, variables, expected_message in error_tests:
    try:
        expr.evaluate(source, variables)
        raise AssertionError(f'Se esperaba error en {source!r}')
    except (ExprError, SyntaxError) as error:
        assert expected_message in str(error), (source, error)

print(f'{len(tests) + len(error_tests)} pruebas superadas')

13 pruebas superadas


## 10. Ejercicios propuestos

1. Agregue potencia `**`, asociativa por la derecha y con mayor precedencia que `*`.
2. Añada literales booleanos `true` y `false`, traducidos a `1` y `0`.
3. Extienda el evaluador con funciones `abs(x)`, `min(a,b)` y `max(a,b)`.
4. Haga que la división entre dos enteros tenga la semántica que haya definido formalmente para MiniTcl.
5. Integre `MiniTclExpr` con el comando `if` del intérprete.
6. Compare los errores producidos por este subconjunto con los de `tclsh`.

### Decisión de diseño para el proyecto

Use dos niveles:

- **Lexer principal de MiniTcl:** separa comandos y argumentos, respetando llaves y comillas.
- **`ExprLexer` + `ExprParser`:** reciben únicamente el argumento de `expr` y procesan su lenguaje interno.

SLY puede emplearse en ambos niveles, pero deben ser clases independientes porque reconocen lenguajes y contextos distintos.